In [1]:
%pip install -q lightgbm scikit-learn pandas numpy


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

TRAIN_DATA = pd.read_csv('train-data.csv', index_col='id')
TRAIN_LABEL = pd.read_csv('train-label.csv', index_col='id')
TEST_DATA = pd.read_csv('test-data.csv', index_col='id')

In [3]:
import numpy as np
import pandas as pd

def preprocess(df):
    df = df.copy()

    # Drop zero-variance and high-cardinality text cols
    drop_cols = [
        'first_name', 'last_name',
        'insitute_name', 'institute_location',
        'test_1', 'test_2', 'test_3', 'test_4', 'test_5',
        'treatment_consent'
    ]
    df = df.drop(columns=drop_cols)

    # === Missingness features FIRST (before any encoding overwrites NaN) ===
    # Class 9 has systematically higher missing rate — this IS a signal
    miss_cols = [
        'gender', 'maternal_defect', 'mother_age', 'father_age',
        'respiration', 'heart_rate', 'risk_level', 'place_birth',
        'folic_acid', 'maternal_illness', 'infertility_treatment',
        'problem_previous_pregnancies', 'abortion_cnt',
        'birth_defects', 'white_blood_cell_count', 'blood_test',
        'symptom_1', 'symptom_2', 'symptom_3', 'symptom_4', 'symptom_5'
    ]
    df['missing_count']      = df[miss_cols].isna().sum(axis=1)
    df['missing_parent_age'] = df['mother_age'].isna().astype(int) + df['father_age'].isna().astype(int)
    df['missing_symptoms']   = df[['symptom_1','symptom_2','symptom_3','symptom_4','symptom_5']].isna().sum(axis=1)
    df['missing_clinical']   = df[['respiration','heart_rate','risk_level','blood_test']].isna().sum(axis=1)

    # Individual flags for highest-signal missing cols
    for col in ['mother_age', 'father_age', 'maternal_defect', 'gender',
                'risk_level', 'heart_rate', 'respiration', 'abortion_cnt',
                'white_blood_cell_count']:
        df[f'{col}_missing'] = df[col].isna().astype(int)

    # === Encode categoricals ===
    binary_yn = [
        'mother_defect', 'father_defect', 'maternal_defect', 'paternal_defect',
        'alive', 'folic_acid', 'maternal_illness', 'infertility_treatment',
        'problem_previous_pregnancies',
        'symptom_1', 'symptom_2', 'symptom_3', 'symptom_4', 'symptom_5'
    ]
    for col in binary_yn:
        df[col] = df[col].map({'Y': 1, 'N': 0})

    df['respiration']   = df['respiration'].map({'A': 1, 'N': 0})
    df['heart_rate']    = df['heart_rate'].map({'A': 1, 'N': 0})
    df['risk_level']    = df['risk_level'].map({'H': 1, 'L': 0})
    df['place_birth']   = df['place_birth'].map({'I': 1, 'H': 0})
    df['birth_defects'] = df['birth_defects'].map({'S': 1, 'M': 2})
    df['gender']        = df['gender'].map({'M': 0, 'F': 1, 'A': 2})
    df['autopsy']       = df['autopsy'].map({'Y': 1, 'N': 0})

    for col in ['birth_asphyxia', 'radiation_exposure', 'substance_abuse']:
        df[col] = df[col].map({'Y': 1, 'N': 0, 'NR': 2})

    df['blood_test'] = df['blood_test'].map({'N': 0, 'I': 1, 'S': 2, 'A': 3})

    # === Engineered features ===
    defect_cols  = ['mother_defect', 'father_defect', 'maternal_defect', 'paternal_defect']
    symptom_cols = ['symptom_1', 'symptom_2', 'symptom_3', 'symptom_4', 'symptom_5']

    df['defect_sum']       = df[defect_cols].sum(axis=1)
    df['symptom_sum']      = df[symptom_cols].sum(axis=1)
    df['defect_x_symptom'] = df['defect_sum'] * df['symptom_sum']
    df['any_defect']       = (df['defect_sum'] > 0).astype(int)
    df['any_symptom']      = (df['symptom_sum'] > 0).astype(int)
    df['high_symptom']     = (df['symptom_sum'] >= 4).astype(int)
    df['all_defects']      = (df['defect_sum'] == 4).astype(int)
    df['parent_age_gap']   = (df['father_age'] - df['mother_age']).abs()

    # Ratio: symptom / (defect+1) — helps distinguish overlap classes
    df['symptom_defect_ratio'] = df['symptom_sum'] / (df['defect_sum'] + 1)

    return df

X_train = preprocess(TRAIN_DATA)
X_test  = preprocess(TEST_DATA)
y_train = TRAIN_LABEL['disorder'].values

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (13249, 53)
X_test shape: (8834, 53)


In [4]:
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
import numpy as np
import pandas as pd

N_SPLITS = 10
SEEDS    = [42, 7, 123]   # average over 3 seeds for stability

all_oof_preds  = np.zeros(len(y_train))
all_test_preds = np.zeros((len(X_test), 10))

for SEED in SEEDS:
    print(f"\n{'='*40}")
    print(f"SEED = {SEED}")
    print('='*40)

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    # --- OOF ETRF (proper, no leakage) ---
    et_oof_train = np.zeros((len(X_train), 10))
    rf_oof_train = np.zeros((len(X_train), 10))
    et_oof_test  = np.zeros((len(X_test),  10))
    rf_oof_test  = np.zeros((len(X_test),  10))

    X_tr_f = X_train.fillna(-1)
    X_te_f = X_test.fillna(-1)

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_tr_f, y_train)):
        Xtr, Xval = X_tr_f.iloc[tr_idx], X_tr_f.iloc[val_idx]
        ytr = y_train[tr_idx]

        et = ExtraTreesClassifier(n_estimators=200, random_state=SEED,
                                   n_jobs=-1, class_weight='balanced')
        rf = RandomForestClassifier(n_estimators=200, random_state=SEED,
                                     n_jobs=-1, class_weight='balanced')
        et.fit(Xtr, ytr)
        rf.fit(Xtr, ytr)

        et_oof_train[val_idx] = et.predict_proba(Xval)
        rf_oof_train[val_idx] = rf.predict_proba(Xval)
        et_oof_test  += et.predict_proba(X_te_f) / N_SPLITS
        rf_oof_test  += rf.predict_proba(X_te_f) / N_SPLITS

    et_cols = [f'et_{i}' for i in range(10)]
    rf_cols = [f'rf_{i}' for i in range(10)]

    X_train_aug = pd.concat([
        X_train.reset_index(drop=True),
        pd.DataFrame(et_oof_train, columns=et_cols),
        pd.DataFrame(rf_oof_train, columns=rf_cols)
    ], axis=1)

    X_test_aug = pd.concat([
        X_test.reset_index(drop=True),
        pd.DataFrame(et_oof_test,  columns=et_cols),
        pd.DataFrame(rf_oof_test,  columns=rf_cols)
    ], axis=1)

    # --- LightGBM folds ---
    oof_preds  = np.zeros(len(y_train))
    test_preds = np.zeros((len(X_test_aug), 10))
    fold_scores = []

    lgbm_params = dict(
        n_estimators    = 3000,
        learning_rate   = 0.01,     # slow = better generalization
        num_leaves      = 31,
        max_depth       = 6,
        subsample       = 0.8,
        subsample_freq  = 1,
        colsample_bytree= 0.8,
        reg_alpha       = 0.1,
        reg_lambda      = 1.0,
        min_child_samples = 20,
        class_weight    = 'balanced',
        random_state    = SEED,
        n_jobs          = -1,
        verbose         = -1
    )

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_aug, y_train)):
        X_tr, X_val = X_train_aug.iloc[train_idx], X_train_aug.iloc[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]

        model = LGBMClassifier(**lgbm_params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[
                early_stopping(150, verbose=False),
                log_evaluation(False)
            ]
        )

        val_pred = model.predict(X_val)
        score    = balanced_accuracy_score(y_val, val_pred)
        fold_scores.append(score)
        print(f"  Fold {fold+1}: BA={score:.4f} (iter={model.best_iteration_})")

        oof_preds[val_idx]  = val_pred
        test_preds         += model.predict_proba(X_test_aug) / N_SPLITS

    oof_score = balanced_accuracy_score(y_train, oof_preds)
    print(f"  OOF BA (seed={SEED}): {oof_score:.4f}")

    all_oof_preds  = all_oof_preds + oof_preds / len(SEEDS)
    all_test_preds = all_test_preds + test_preds / len(SEEDS)

# Final OOF (rounded average)
final_oof = np.round(all_oof_preds).astype(int)
final_oof_score = balanced_accuracy_score(y_train, final_oof)
print(f"\nFinal averaged OOF BA: {final_oof_score:.4f}")


SEED = 42
  Fold 1: BA=0.3337 (iter=811)
  Fold 2: BA=0.2906 (iter=790)
  Fold 3: BA=0.3164 (iter=1058)
  Fold 4: BA=0.2876 (iter=1060)
  Fold 5: BA=0.3114 (iter=1089)
  Fold 6: BA=0.3054 (iter=952)
  Fold 7: BA=0.2930 (iter=1075)
  Fold 8: BA=0.2889 (iter=882)
  Fold 9: BA=0.3431 (iter=1007)
  Fold 10: BA=0.3334 (iter=1021)
  OOF BA (seed=42): 0.3102

SEED = 7
  Fold 1: BA=0.3270 (iter=1080)
  Fold 2: BA=0.2948 (iter=1158)
  Fold 3: BA=0.3094 (iter=744)
  Fold 4: BA=0.3563 (iter=880)
  Fold 5: BA=0.3102 (iter=1077)
  Fold 6: BA=0.3058 (iter=1010)
  Fold 7: BA=0.3301 (iter=1098)
  Fold 8: BA=0.2996 (iter=756)
  Fold 9: BA=0.3231 (iter=836)
  Fold 10: BA=0.3217 (iter=1129)
  OOF BA (seed=7): 0.3178

SEED = 123
  Fold 1: BA=0.2971 (iter=959)
  Fold 2: BA=0.3331 (iter=896)
  Fold 3: BA=0.2932 (iter=813)
  Fold 4: BA=0.3218 (iter=871)
  Fold 5: BA=0.3643 (iter=993)
  Fold 6: BA=0.3134 (iter=692)
  Fold 7: BA=0.2799 (iter=966)
  Fold 8: BA=0.3040 (iter=1175)
  Fold 9: BA=0.3364 (iter=916)


In [5]:
final_preds = np.argmax(all_test_preds, axis=1)

submission = pd.DataFrame({
    'id': TEST_DATA.index,
    'disorder': final_preds
}).set_index('id')

submission.to_csv('submission-a2.csv')
print("Saved! Shape:", submission.shape)
print(submission['disorder'].value_counts().sort_index())

Saved! Shape: (8834, 1)
disorder
0     338
1    1745
2     762
3    1963
4      35
5    1311
6     915
7    1102
8      51
9     612
Name: count, dtype: int64
